<a href="https://colab.research.google.com/github/springboardmentor112r-Agri/Oil_Spill_Detection-/blob/Kajal_AI_OIL_SPILL_DETECTION/EDA_Preprocessing_Oil_Spill.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
from google.colab import drive
import os

# 1. Mount Google Drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [8]:
# Copy zip file to Colab
!cp /content/drive/MyDrive/15298010.zip /content/

# Unzip
!unzip /content/15298010.zip -d /content/15298010


Archive:  /content/15298010.zip
 extracting: /content/15298010/images.zip  
 extracting: /content/15298010/masks.zip  


In [9]:
#images and masks are present
!ls /content/15298010


images.zip  masks.zip


In [10]:
!unzip /content/15298010/images.zip -d /content/15298010/images


Streaming output truncated to the last 5000 lines.
  inflating: /content/15298010/images/images/train/sentinel_555.png  
  inflating: /content/15298010/images/__MACOSX/images/train/._sentinel_555.png  
  inflating: /content/15298010/images/images/train/sentinel_2145.png  
  inflating: /content/15298010/images/__MACOSX/images/train/._sentinel_2145.png  
  inflating: /content/15298010/images/images/train/palsar_2465.png  
  inflating: /content/15298010/images/__MACOSX/images/train/._palsar_2465.png  
  inflating: /content/15298010/images/images/train/palsar_2303.png  
  inflating: /content/15298010/images/__MACOSX/images/train/._palsar_2303.png  
  inflating: /content/15298010/images/images/train/sentinel_2623.png  
  inflating: /content/15298010/images/__MACOSX/images/train/._sentinel_2623.png  
  inflating: /content/15298010/images/images/train/sentinel_2637.png  
  inflating: /content/15298010/images/__MACOSX/images/train/._sentinel_2637.png  
  inflating: /content/15298010/images/ima

In [11]:
!unzip /content/15298010/masks.zip -d /content/15298010/masks


Streaming output truncated to the last 5000 lines.
  inflating: /content/15298010/masks/masks/train/sentinel_555.png  
  inflating: /content/15298010/masks/__MACOSX/masks/train/._sentinel_555.png  
  inflating: /content/15298010/masks/masks/train/sentinel_2145.png  
  inflating: /content/15298010/masks/__MACOSX/masks/train/._sentinel_2145.png  
  inflating: /content/15298010/masks/masks/train/palsar_2465.png  
  inflating: /content/15298010/masks/__MACOSX/masks/train/._palsar_2465.png  
  inflating: /content/15298010/masks/masks/train/palsar_2303.png  
  inflating: /content/15298010/masks/__MACOSX/masks/train/._palsar_2303.png  
  inflating: /content/15298010/masks/masks/train/sentinel_2623.png  
  inflating: /content/15298010/masks/__MACOSX/masks/train/._sentinel_2623.png  
  inflating: /content/15298010/masks/masks/train/sentinel_2637.png  
  inflating: /content/15298010/masks/__MACOSX/masks/train/._sentinel_2637.png  
  inflating: /content/15298010/masks/masks/train/palsar_2317.png 

In [12]:
#ensure that 'images' and 'masks' folders (with train/val splits) exist correctly
!find /content/15298010 -maxdepth 2 -type d


/content/15298010
/content/15298010/images
/content/15298010/images/__MACOSX
/content/15298010/images/images
/content/15298010/masks
/content/15298010/masks/__MACOSX
/content/15298010/masks/masks


In [13]:
#Remove macOS metadata folders (__MACOSX) and redundant nested directories
# created during ZIP extraction to clean and flatten the dataset structure
!rm -rf /content/15298010/images/__MACOSX
!rm -rf /content/15298010/masks/__MACOSX

!rm -rf /content/15298010/images/images
!rm -rf /content/15298010/masks/masks


In [14]:
!find /content/15298010 -maxdepth 2 -type d


/content/15298010
/content/15298010/images
/content/15298010/masks


In [15]:
!ls /content/15298010/images | head
!ls /content/15298010/masks | head


In [16]:
!ls /content/15298010/images/train | head
!ls /content/15298010/masks/train | head

!ls /content/15298010/images/val | head
!ls /content/15298010/masks/val | head


ls: cannot access '/content/15298010/images/train': No such file or directory
ls: cannot access '/content/15298010/masks/train': No such file or directory
ls: cannot access '/content/15298010/images/val': No such file or directory
ls: cannot access '/content/15298010/masks/val': No such file or directory


In [17]:
#sanity check
import cv2
import matplotlib.pyplot as plt
import numpy as np
import os

img_dir = "/content/15298010/images/train"
mask_dir = "/content/15298010/masks/train"

img_files = sorted(os.listdir(img_dir))
mask_files = sorted(os.listdir(mask_dir))

idx = np.random.randint(0, len(img_files))

img = cv2.imread(os.path.join(img_dir, img_files[idx]))
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

mask = cv2.imread(os.path.join(mask_dir, mask_files[idx]), 0)

plt.figure(figsize=(10,4))
plt.subplot(1,2,1)
plt.imshow(img)
plt.title("Train Image")
plt.axis("off")

plt.subplot(1,2,2)
plt.imshow(mask, cmap="gray")
plt.title("Train Mask")
plt.axis("off")

plt.show()


FileNotFoundError: [Errno 2] No such file or directory: '/content/15298010/images/train'

In [ ]:
!pip install -q albumentations segmentation-models-pytorch


In [ ]:
import os
import cv2
import numpy as np
import albumentations as A
from albumentations.pytorch import ToTensorV2

import torch
from torch.utils.data import Dataset, DataLoader


In [ ]:
BASE_PATH = "/content/15298010"

TRAIN_IMG = os.path.join(BASE_PATH, "images/train")
TRAIN_MASK = os.path.join(BASE_PATH, "masks/train")

VAL_IMG = os.path.join(BASE_PATH, "images/val")
VAL_MASK = os.path.join(BASE_PATH, "masks/val")


In [ ]:
class OilSpillDataset(Dataset):
    def __init__(self, image_dir, mask_dir, transform=None):
        self.image_dir = image_dir
        self.mask_dir = mask_dir
        self.images = sorted(os.listdir(image_dir))
        self.masks = sorted(os.listdir(mask_dir))
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_path = os.path.join(self.image_dir, self.images[idx])
        mask_path = os.path.join(self.mask_dir, self.masks[idx])

        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        mask = cv2.imread(mask_path, 0)
        mask = (mask > 0).astype(np.float32)

        if self.transform:
            augmented = self.transform(image=image, mask=mask)
            image = augmented["image"]
            mask = augmented["mask"].unsqueeze(0)

        return image, mask


In [ ]:
train_transform = A.Compose([
    A.Resize(256, 256),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.Normalize(),
    ToTensorV2()
])

val_transform = A.Compose([
    A.Resize(256, 256),
    A.Normalize(),
    ToTensorV2()
])


In [ ]:
train_dataset = OilSpillDataset(TRAIN_IMG, TRAIN_MASK, train_transform)
val_dataset   = OilSpillDataset(VAL_IMG, VAL_MASK, val_transform)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=8, shuffle=False)

print("DataLoaders ready")


In [ ]:
import os

def count_files(path):
    return len(os.listdir(path))

print("Train Images:", count_files("/content/15298010/images/train"))
print("Train Masks :", count_files("/content/15298010/masks/train"))

print("Val Images  :", count_files("/content/15298010/images/val"))
print("Val Masks   :", count_files("/content/15298010/masks/val"))


In [ ]:
import os
import cv2
import numpy as np


In [ ]:
TRAIN_MASK_DIR = "/content/15298010/masks/train"
VAL_MASK_DIR   = "/content/15298010/masks/val"


In [ ]:
def count_oil_non_oil(mask_dir):
    oil = 0
    non_oil = 0

    for mask_file in os.listdir(mask_dir):
        mask_path = os.path.join(mask_dir, mask_file)
        mask = cv2.imread(mask_path, 0)

        # Check if any oil pixel exists
        if np.any(mask > 0):
            oil += 1
        else:
            non_oil += 1

    return oil, non_oil


In [ ]:
train_oil, train_non_oil = count_oil_non_oil(TRAIN_MASK_DIR)
val_oil, val_non_oil     = count_oil_non_oil(VAL_MASK_DIR)

print("🔹 TRAIN SET")
print("Oil Spill Images     :", train_oil)
print("Non-Oil Images       :", train_non_oil)

print("\n🔹 VALIDATION SET")
print("Oil Spill Images     :", val_oil)
print("Non-Oil Images       :", val_non_oil)

print("\n🔹 TOTAL DATASET")
print("Oil Spill Images     :", train_oil + val_oil)
print("Non-Oil Images       :", train_non_oil + val_non_oil)


In [ ]:

sizes = []

for img in os.listdir(TRAIN_IMG):
    img_path = os.path.join(TRAIN_IMG, img)
    image = cv2.imread(img_path)
    h, w, _ = image.shape
    sizes.append((h, w))

print("Unique image sizes:", set(sizes))


In [ ]:
import numpy as np

oil_pixels = []
total_pixels = []

for mask_file in os.listdir(TRAIN_MASK):
    mask = cv2.imread(os.path.join(TRAIN_MASK, mask_file), 0)
    oil_pixels.append(np.sum(mask > 0))
    total_pixels.append(mask.size)

oil_percentage = np.array(oil_pixels) / np.array(total_pixels)

print("Average oil coverage:", np.mean(oil_percentage) * 100, "%")


In [ ]:
import matplotlib.pyplot as plt

labels = ['Oil Spill', 'Non-Oil']
counts = [train_oil + val_oil, train_non_oil + val_non_oil]

plt.bar(labels, counts)
plt.title("Oil vs Non-Oil Distribution")
plt.show()


In [ ]:
images, masks = next(iter(train_loader))

plt.figure(figsize=(12,4))
for i in range(3):
    plt.subplot(2,3,i+1)
    plt.imshow(images[i].permute(1,2,0))
    plt.title("Augmented Image")
    plt.axis("off")

    plt.subplot(2,3,i+4)
    plt.imshow(masks[i][0], cmap="gray")
    plt.title("Augmented Mask")
    plt.axis("off")

plt.show()


In [ ]:
import matplotlib.pyplot as plt
import cv2
import numpy as np
import os

def get_oil_and_non_oil(mask_dir, img_dir):
    oil_img = non_oil_img = None

    for file in os.listdir(mask_dir):
        mask = cv2.imread(os.path.join(mask_dir, file), 0)
        img = cv2.imread(os.path.join(img_dir, file))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        if np.any(mask > 0) and oil_img is None:
            oil_img = img
        elif not np.any(mask > 0) and non_oil_img is None:
            non_oil_img = img

        if oil_img is not None and non_oil_img is not None:
            break

    return oil_img, non_oil_img

oil_img, non_oil_img = get_oil_and_non_oil(
    "/content/15298010/masks/train",
    "/content/15298010/images/train"
)

plt.figure(figsize=(10,4))
plt.subplot(1,2,1)
plt.imshow(oil_img)
plt.title("Oil Spill Image")
plt.axis("off")

plt.subplot(1,2,2)
plt.imshow(non_oil_img)
plt.title("Non-Oil Image")
plt.axis("off")

plt.show()


In [ ]:
import numpy as np

coverages = []

for mask_file in os.listdir("/content/15298010/masks/train"):
    mask = cv2.imread(
        os.path.join("/content/15298010/masks/train", mask_file), 0
    )
    coverage = np.sum(mask > 0) / mask.size
    coverages.append(coverage)

plt.hist(coverages, bins=30)
plt.title("Oil Spill Area Distribution")
plt.xlabel("Oil Coverage Ratio")
plt.ylabel("Number of Images")
plt.show()
